# Router Stage 2 -- Modality Classifier (CliniqAI)

**Task:** given an image already confirmed medical by Stage 1, which of your 6 modalities is it?

**Classes:** `brain_mri`, `kidney_ct`, `retinal`, `skin_lesion`, `pneumonia`, `breast_histology`

Trained only on medical images -- same 6 dataset folders as Stage 1's positive class, but now
each image is labeled by *which* modality it came from instead of a single `medical` label.

**Why this is an easy task:** fundus photos, CT slices, chest X-rays, dermoscopy images, and
histology patches are visually very distinct domains -- expect 95-100%, similar to your earlier
3-class flat router.

**Extending later:** once Roneet's `bone_fracture` specialist is ready, add its folder to
`MODALITY_SOURCES` below and rerun this notebook -- it becomes a 7-class problem with no other
code changes needed.

**Config (matches your specialists):** EfficientNet-B0, batch size 32, lr 1e-3, weight decay 1e-4,
early stopping patience 10, 224x224 input, `num_workers=0` (Windows).


## 1. Session setup

In [ ]:
import os
import sys

PROJECT_ROOT = r"E:\programming\CliniqAI"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())

## 2. Configure dataset roots

**Check these paths before running.** To add `bone_fracture` later, add one line here --
everything else in this notebook adapts automatically since it reads `len(MODALITY_SOURCES)`
for the model's output size.

In [ ]:
from pathlib import Path

# Modality name -> root folder (searched recursively). Add new entries here to extend.
MODALITY_SOURCES = {
    "brain_mri": Path("data/raw/brain_tumor"),
    "kidney_ct": Path("data/raw/kidney_stone"),
    "retinal": Path("data/raw/diabetic_retinopathy"),
    "skin_lesion": Path("data/raw/skin_lesion"),
    "pneumonia": Path("data/raw/pneumonia"),
    "breast_histology": Path("data/raw/breast_histology"),
    # "bone_fracture": Path("data/raw/bone_fracture"),  # uncomment once Roneet's data is ready
}

for name, path in MODALITY_SOURCES.items():
    status = "OK" if path.exists() else "MISSING -- fix this path"
    print(f"  {name:<20} {str(path):<45} {status}")

CLASS_NAMES = sorted(MODALITY_SOURCES.keys())
print(f"\nClasses ({len(CLASS_NAMES)}): {CLASS_NAMES}")

## 3. Collect image paths per modality

Internal class/patient subfolder structure within each dataset is ignored -- every image under a modality's root is pooled under that modality's single label.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
MAX_PER_CLASS = 4000  # caps large datasets (e.g. breast_histology) so no class dominates training
RANDOM_SEED = 42

import random
random.seed(RANDOM_SEED)

def collect_images(root, max_count=None):
    if not root.exists():
        print(f"  [skip] {root} does not exist")
        return []
    files = [p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
    if max_count and len(files) > max_count:
        files = random.sample(files, max_count)
    return files

records = []
for modality, root in MODALITY_SOURCES.items():
    files = collect_images(root, MAX_PER_CLASS)
    print(f"  {modality:<20} {len(files)} images")
    records.extend([(str(p), modality) for p in files])

print(f"\nTotal images: {len(records)}")

## 4. Build dataframe and split (70/15/15, image-level)

Same reasoning as Stage 1 -- modality recognition doesn't need patient-level splitting.

In [ ]:
import pandas as pd

df = pd.DataFrame(records, columns=["filepath", "label"])
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

n = len(df)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_df = df.iloc[:n_train].reset_index(drop=True)
val_df = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
test_df = df.iloc[n_train + n_val:].reset_index(drop=True)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print()
print(train_df["label"].value_counts())

## 5. Dataset, transforms, dataloaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 224
BATCH_SIZE = 32

label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
idx_to_label = {i: name for name, i in label_to_idx.items()}

class ModalityDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.paths = dataframe["filepath"].tolist()
        self.labels = [label_to_idx[l] for l in dataframe["label"].tolist()]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = ModalityDataset(train_df, transform=train_transform)
val_dataset = ModalityDataset(val_df, transform=eval_transform)
test_dataset = ModalityDataset(test_df, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches -> train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")
print("Class index mapping:", idx_to_label)

## 6. Model -- EfficientNet-B0 (N-class head)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

NUM_CLASSES = len(CLASS_NAMES)

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

## 7. Training loop with early stopping (patience=10)

In [ ]:
import time
from pathlib import Path

CHECKPOINT_DIR = Path("models/router")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "router_stage2_efficientnet_b0.pt"

NUM_EPOCHS = 20
PATIENCE = 10

best_val_loss = float("inf")
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(train_mode)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if train_mode:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if train_mode:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    elapsed = time.time() - start

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
          f"{elapsed:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
            "epoch": epoch,
            "class_names": CLASS_NAMES,
        }, CHECKPOINT_PATH)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

print("\nTraining complete. Best val_loss:", best_val_loss)

## 8. Evaluate on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Test accuracy:", (np.array(all_preds) == np.array(all_labels)).mean())
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Router Stage 2 -- Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Notes

- **Checkpoint:** `models/checkpoints/router_stage2_efficientnet_b0.pt` -- sync to Drive same as
  your specialists. `class_names` is saved inside the checkpoint so your backend's model registry
  can read the class order directly instead of hardcoding it.
- **Combined pipeline accuracy** is roughly Stage1_acc x Stage2_acc x Specialist_acc -- if each
  stage is genuinely 95%+, your end-to-end pipeline stays comfortably in a usable range even
  compounding three models' worth of error.
- **Adding bone_fracture:** uncomment its line in `MODALITY_SOURCES` (cell 2) once Roneet's
  dataset is ready, rerun this whole notebook. The class count, model head size, and confusion
  matrix all adapt automatically -- no other code changes.
- This checkpoint is what your backend's Stage 2 step loads to decide which specialist
  (`models/checkpoints/<modality>_specialist_v1.pt`) to route an image to.
